In [24]:
import pickle
import numpy as np
import os
import pprint

# file = open('inputs/wilson/random_weekeday_2.pkl', 'rb')
file = open('inputs/localDB_payload_oct.pkl', 'rb')
data = pickle.load(file)
file.close()

In [25]:
print(data)

{'driver_runs': [{'run_id': 0, 'start_time': 14400, 'end_time': 54000, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 1, 'start_time': 14400, 'end_time': 54000, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 2, 'start_time': 18000, 'end_time': 54000, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 3, 'start_time': 18000, 'end_time': 54000, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 4, 'start_time': 25200, 'end_time': 64800, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 5, 'start_time': 25200, 'end_time': 64800, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 6, 'start_time': 25200, 'end_time': 64800, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 7, 'start_time': 25200, 'end_time': 64800, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 8, 'start_time': 25200, 'end_time': 72000, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 9, 'start_time': 25200, 'end_time': 72000, 'am_capacity': 8, 'wc_capacity': 3}, {'run_id': 10, 'start_time': 54000, 'end_time': 79200, 'am_capacity': 8, 

In [26]:

pprint.pprint(data, depth=2)

{'date': '2023-10-19',
 'depot': {'node_id': 0, 'pt': {...}},
 'driver_runs': [{...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...},
                 {...}],
 'requests': [{...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
              {...},
 

In [31]:
pickup_ids = []
dropoff_ids = []
for request in data['requests']:
    pickup = request['pickup_node_id']
    dropoff = request['dropoff_node_id']
    print(pickup, dropoff)
    pickup_ids.append(pickup)
    dropoff_ids.append(dropoff)
print("Unique pickup ids:", set(pickup_ids), "and length:", len(set(pickup_ids)))
print("Unique dropoff ids:", set(dropoff_ids), "and length:", len(set(dropoff_ids)))

1 2
3 4
5 6
7 8
9 10
11 12
13 14
15 16
17 18
19 20
21 22
23 24
25 26
27 28
29 30
31 32
33 34
35 36
37 38
39 40
41 42
43 44
45 46
47 48
49 50
51 52
53 54
55 56
57 58
59 60
61 62
63 64
65 66
67 68
69 70
71 72
73 74
75 76
77 78
79 80
81 82
83 84
85 86
87 88
89 90
91 92
93 94
95 96
97 98
99 100
101 102
103 104
105 106
107 108
109 110
111 112
113 114
115 116
117 118
119 120
121 122
123 124
125 126
127 128
129 130
131 132
133 134
135 136
137 138
139 140
141 142
143 144
145 146
147 148
149 150
151 152
153 154
155 156
157 158
159 160
161 162
163 164
165 166
167 168
169 170
171 172
173 174
175 176
177 178
179 180
181 182
183 184
185 186
187 188
189 190
191 192
193 194
195 196
197 198
199 200
201 202
203 204
205 206
207 208
209 210
211 212
213 214
215 216
217 218
219 220
221 222
223 224
225 226
227 228
229 230
231 232
233 234
235 236
237 238
239 240
241 242
243 244
245 246
247 248
249 250
251 252
253 254
255 256
257 258
259 260
261 262
263 264
265 266
267 268
269 270
271 272
273 274
275 276
277 

In [3]:
from rtv_solver import OnlineRTVSolver

# Initialize the RTV solver with the URL of the OSRM server
online_rtv_solver = OnlineRTVSolver("http://127.0.0.1:5001/")

In [ ]:
# creating a new payload with new requests
# consider all requests that start before 05:40:00

current_time = 5*3600+30*60
step = 10*60

selected_requests = []
for request in data["requests"]:
    if request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

3

In [ ]:
# create a new payload with the selected requests
new_payload = {
    "depot": data["depot"],
    "requests": selected_requests,
    "driver_runs": data["driver_runs"],}

## Fast Heuristic method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_heuristic(new_payload)
unserved_requests

[]

In [6]:
# Simulate to 5:40:00

current_time += step
simulated_driver_runs = online_rtv_solver.simulate_manifest(current_time,new_driver_runs,intermediate_location=False)

In [ ]:
# creating a new payload with new requests
# consider all requests that start between 05:40:00 and 05:50:00

selected_requests = []
for request in data["requests"]:
    if request["pickup_time_window_start"] >= current_time and request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

2

In [ ]:
# create a new payload with the selected requests
new_payload = {
    "depot": data["depot"],
    "requests": selected_requests,
    "driver_runs": simulated_driver_runs,}

## Full RTV method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_rtv(new_payload)
unserved_requests

[]

In [9]:
# Simulate to 5:50:00

current_time += step
simulated_driver_runs = online_rtv_solver.simulate_manifest(current_time,new_driver_runs,intermediate_location=False)

In [ ]:
# creating a new payload with new requests
# consider all requests that are between before 05:50:00 and after 05:60:00

selected_requests = []
for request in data["requests"]:
    if request["pickup_time_window_start"] >= current_time and request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

3

In [ ]:
req = selected_requests[0]

# check feasibility of time slots


new_payload = {
    "depot": data["depot"], # JW: added to get it running
    "requests": [
    {
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'time_windows' : [
            {'pickup_time_window_start': req['pickup_time_window_start'], 'pickup_time_window_end': req['pickup_time_window_start'] + 60, 'dropoff_time_window_start': req['dropoff_time_window_start'], 'dropoff_time_window_end': req['dropoff_time_window_start'] + 180},
            {'pickup_time_window_start': req['pickup_time_window_start']+900, 'pickup_time_window_end': req['pickup_time_window_end']+900, 'dropoff_time_window_start': req['dropoff_time_window_start']+900, 'dropoff_time_window_end': req['dropoff_time_window_end']+900},
            {'pickup_time_window_start': req['pickup_time_window_start']+1800, 'pickup_time_window_end': req['pickup_time_window_end']+1800, 'dropoff_time_window_start': req['dropoff_time_window_start']+1800, 'dropoff_time_window_end': req['dropoff_time_window_end']+1800},
        ],
        'am': req['am'],
        'wc': req['wc']
    }],
    "driver_runs": simulated_driver_runs
}


feasible_windows = online_rtv_solver.check_feasibility(new_payload)
feasible_windows

[({'pickup_time_window_start': 21966,
   'pickup_time_window_end': 23766,
   'dropoff_time_window_start': 22661,
   'dropoff_time_window_end': 24461},
  1.1359886201991465),
 ({'pickup_time_window_start': 22866,
   'pickup_time_window_end': 24666,
   'dropoff_time_window_start': 23561,
   'dropoff_time_window_end': 25361},
  1.1524893314367)]

In [12]:
feasible_windows[0] # this has the feasible time slot and associated VMT/PMT ratio for this slot

({'pickup_time_window_start': 21966,
  'pickup_time_window_end': 23766,
  'dropoff_time_window_start': 22661,
  'dropoff_time_window_end': 24461},
 1.1359886201991465)

In [ ]:
# Creating a request with infeasible time slots

req = selected_requests[0]

new_payload = {
    "depot": data["depot"],
    "requests": [
    {
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'pickup_time_window_start': req['pickup_time_window_start'], 
        'pickup_time_window_end': req['pickup_time_window_start']+180, 
        'dropoff_time_window_start': req['dropoff_time_window_start'], 
        'dropoff_time_window_end': req['dropoff_time_window_start']+180,
        'am': req['am'],
        'wc': req['wc']
    }],
    "driver_runs": simulated_driver_runs
}

## Full RTV method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_rtv(new_payload)
unserved_requests

['6']

In [14]:
# Serve at earliest possible time

# JW: API needs to fit the output, "new_unserved_requests" must be binded otherwise it breaks the next run
new_driver_runs, new_unserved_requests = online_rtv_solver.serve_asap(new_payload)

In [ ]:
# Reoptimize the driver runs
# JW: optional: payload is incorrect

repotimized_driver_runs = online_rtv_solver.resolve_pdptw_rtv({"depot": data["depot"], "driver_runs": new_driver_runs, "requests": new_unserved_requests})